In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.labelsize":     9,
    "axes.titlesize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "figure.dpi":         150,
})

TEAL = ( 30/255, 130/255, 130/255)
RED  = (177/255,  56/255,  69/255)

PANELS = [
    (0.85, TEAL),
    (0.95, RED),
]

def _bar_color(rgb, x, x_max, pale_mix=0.72):
    t = min(abs(x) / x_max, 1.0) if x_max > 0 else 0.0
    return tuple(c + (1 - c) * pale_mix * t for c in rgb)

def _edge_color(rgb, x, x_max, pale_mix=0.35):
    t = min(abs(x) / x_max, 1.0) if x_max > 0 else 0.0
    return tuple(c + (1 - c) * pale_mix * t for c in rgb)

def simulate_erw_batch(n, p, q, n_sims, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    xi = np.zeros((n_sims, n + 1), dtype=np.int8)
    xi[:, 1] = np.where(rng.random(n_sims) < q, 1, -1)
    for t in range(1, n):
        idx  = rng.integers(1, t + 1, size=n_sims)
        past = xi[np.arange(n_sims), idx]
        flip = np.where(rng.random(n_sims) < p, 1, -1)
        xi[:, t + 1] = past * flip
    return xi[:, 1:].sum(axis=1).astype(float)

N_STEPS = 10_000
N_SIMS  = 10_000
N_BINS  = 52
Q       = 0.50

rng = np.random.default_rng(2026)

finals = {}
for p_val, _ in PANELS:
    finals[p_val] = simulate_erw_batch(N_STEPS, p=p_val, q=Q, n_sims=N_SIMS, rng=rng)

panel_xlim   = {}
panel_edges  = {}
panel_counts = {}
panel_ylim   = {}

for p_val, _ in PANELS:
    xlim = float(np.quantile(np.abs(finals[p_val]), 0.995)) * 1.05
    xlim = np.ceil(xlim / 500) * 500
    panel_xlim[p_val] = xlim

    edges = np.linspace(-xlim, xlim, N_BINS + 1)
    panel_edges[p_val] = edges

    c, _ = np.histogram(finals[p_val], bins=edges, density=True)
    panel_counts[p_val] = c
    panel_ylim[p_val] = c.max() * 1.08

fig, axes = plt.subplots(1, 2, figsize=(5.2, 3.0))
fig.subplots_adjust(left=0.13, right=0.98, bottom=0.19, top=0.83, wspace=0.35)

LABEL_GREY = (0.55, 0.55, 0.55)

def _nice_xticks(xlim):
    t = xlim * 0.75

    t = np.round(t / 500) * 500
    if t <= 0:
        t = np.round(xlim * 0.5 / 500) * 500
    return [-t, 0.0, t]

for ax, (p_val, color) in zip(axes, PANELS):
    xlim    = panel_xlim[p_val]
    ylim    = panel_ylim[p_val]
    edges   = panel_edges[p_val]
    counts  = panel_counts[p_val]
    width   = edges[1] - edges[0]
    centers = 0.5 * (edges[:-1] + edges[1:])
    xticks  = _nice_xticks(xlim)

    for xc, h in zip(centers, counts):
        ax.bar(xc, h, width=width,
               color=_bar_color(color, xc, xlim),
               edgecolor=_edge_color(color, xc, xlim),
               linewidth=0.28, alpha=0.70, zorder=2)

    ax.set_title(fr"$p\!=\!{p_val:.2f}$", fontsize=9, pad=14, color=LABEL_GREY)
    ax.set_xlabel(r"$S_n$", labelpad=3)
    ax.set_xlim(-xlim, xlim)
    ax.set_ylim(0, ylim)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.55)
    ax.spines["bottom"].set_linewidth(0.55)
    ax.tick_params(width=0.55, length=3, direction="out")

    ax.xaxis.set_major_locator(ticker.FixedLocator(xticks))
    def _fmt(t, _):
        if t == 0:
            return r"$0$"

        return fr"${int(t):d}$"
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(_fmt))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3))

    y_formatter = ticker.ScalarFormatter(useMathText=True)
    y_formatter.set_scientific(True)
    y_formatter.set_powerlimits((-2, 2))
    ax.yaxis.set_major_formatter(y_formatter)
    ax.yaxis.get_offset_text().set_fontsize(7)
    ax.yaxis.get_offset_text().set_color(LABEL_GREY)

axes[0].set_ylabel("Density", labelpad=4, fontsize=8)

fig.savefig("figure_4_5.pdf", bbox_inches="tight", pad_inches=0.02)
print("Saved figure_4_5.pdf")